In [ ]:
from enum import Enum

class Categorie(str, Enum):
	"""Categorie disponibili per classificare una transazione. Se è un'entrata classificalla come Entrata, se è una spesa classificalla come una delle altre categorie."""

	AFFITTO_O_MUTUO = "Affitto o Mutuo"
	CONDOMINIO = "Condominio"
	ACQUA = "Acqua"
	GAS = "Gas"
	LUCE = "Luce"
	TELEFONO_INTERNET = "Telefono / Internet"
	ASSICURAZIONI = "Assicurazioni"
	MACCHINA = "Macchina"
	TASSE = "Tasse"
	PALESTRA = "Palestra"
	ABBONAMENTI = "Abbonamenti"
	ALIMENTAZIONE = "Alimentazione"
	SPESE_MEDICHE_E_FARMACIA = "Spese Mediche e Farmacia"
	TRASPORTO = "Trasporto"
	RISTORANTE = "Ristorante"
	BAR = "Bar"
	DELIVERY = "Delivery"
	CULTURALE = "Culturale"
	VESTITI = "Vestiti"
	FIGLI = "Figli"
	PET = "Pet"
	AMAZON = "Amazon"
	HOBBY = "Hobby"
	PRELIEVI_BANCOMAT = "Prelievi Bancomat"
	SATISPAY = "Satispay"
	ENTRATA = "Entrata"
	ALTRO = "Altro"


In [ ]:
from datetime import date as date_type
from pathlib import Path

from pydantic import BaseModel, Field
from sqlmodel import Field as SQLField, SQLModel, Session, create_engine

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "db").exists() else Path.cwd().parent
DB_DIR = PROJECT_ROOT / "db"
DB_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_PATH = DB_DIR / "bilancio.sqlite3"
DATABASE_URL = f"sqlite:///{DATABASE_PATH}"
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False},
    echo=False,
 )

class MovimentoRaw(BaseModel):
    class Data(BaseModel):
        giorno: int = Field(description="Giorno dell'operazione")
        mese: int = Field(description="Mese dell'operazione")
        anno: int = Field(description="Anno dell'operazione")

    data: Data = Field(description="Data dell'operazione")
    descrizione: str = Field(description="Descrizione dell'operazione")
    importo: float = Field(description="Importo dell'operazione")
    note: str = Field(description="Informazioni aggiuntive sull'operazione")

class MovimentoInput(BaseModel):
    data: date_type = Field(description="Data dell'operazione")
    descrizione: str = Field(description="Descrizione dell'operazione")
    importo: float = Field(description="Importo dell'operazione")
    categoria: Categorie = Field(description="Categoria dell'operazione")
    note: str | None = Field(default=None, description="Informazioni aggiuntive sull'operazione")

class MovimentoPrimary(SQLModel, table=True):
    __tablename__ = "movimenti"
    __table_args__ = {"extend_existing": True}

    id: int | None = SQLField(default=None, primary_key=True)
    data: date_type = SQLField(index=True, description="Data dell'operazione")
    descrizione: str = SQLField(index=True, description="Descrizione dell'operazione")
    importo: float = SQLField(description="Importo dell'operazione")
    categoria: Categorie = SQLField(description="Categoria dell'operazione")
    note: str | None = SQLField(default=None, description="Informazioni aggiuntive sull'operazione")

def create_db_and_tables() -> None:
    SQLModel.metadata.create_all(engine)

In [ ]:
create_db_and_tables()
DATABASE_PATH

In [ ]:
system_prompt = """# Punkathon Agent
Sei un assistente virtuale che aiuta gli utenti a gestire il proprio bilancio personale. Puoi aggiungere movimenti al bilancio, classificare le spese in categorie e fornire riepiloghi mensili. Utilizza la funzione `aggiungi_movimenti` per aggiungere nuovi movimenti al bilancio. Ogni movimento deve includere una data, una descrizione, un importo e eventuali note aggiuntive.

Puoi ricevere come input testo libero, un DataFrame serializzato in base64 come file CSV o JSON, un'immagine in base64 oppure un PDF in base64.
Quando ricevi file, immagini o PDF devi leggerli, estrarre i movimenti trovati, classificarli usando le categorie disponibili e salvare quelli validi con `aggiungi_movimenti`.
Se il documento non contiene movimenti o mancano dati essenziali, spiega chiaramente il problema invece di inventare valori.
"""

In [ ]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

async def aggiungi_movimenti(movimenti: list[MovimentoInput]) -> str:
    """Salva uno o più movimenti classificati nel database del bilancio."""
    create_db_and_tables()
    movimenti_normalizzati = [
        MovimentoPrimary(**movimento.model_dump())
        for movimento in movimenti
    ]

    with Session(engine) as session:
        session.add_all(movimenti_normalizzati)
        session.commit()

    return f"Ho aggiunto {len(movimenti_normalizzati)} movimenti alla tabella dei movimenti."


model = ChatOpenAI(
    model="gpt-5.4",
    use_responses_api=True,
    output_version="responses/v1",
    reasoning={"summary": "auto"},
)

agent = create_deep_agent(
    model=model,
    tools=[aggiungi_movimenti],
    system_prompt=system_prompt,
 )

In [ ]:
import base64
import mimetypes
from typing import Any

def encode_file_to_base64(path: str | Path) -> str:
    return base64.b64encode(Path(path).read_bytes()).decode("utf-8")

def decode_base64_text(base64_text: str, encoding: str = "utf-8") -> str:
    return base64.b64decode(base64_text).decode(encoding)

def _guess_mime_type(path: str | Path, default: str = "application/octet-stream") -> str:
    guessed, _ = mimetypes.guess_type(str(path))
    return guessed or default

def build_base64_image_content(
    prompt: str,
    *,
    image_base64: str,
    mime_type: str = "image/jpeg",
    detail: str = "auto",
) -> list[dict[str, Any]]:
    return [
        {"type": "text", "text": prompt},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_base64}",
                "detail": detail,
            },
        },
    ]

def build_base64_file_content(
    prompt: str,
    *,
    file_base64: str,
    filename: str,
    mime_type: str,
) -> list[dict[str, Any]]:
    return [
        {"type": "text", "text": prompt},
        {
            "type": "file",
            "file": {
                "file_data": f"data:{mime_type};base64,{file_base64}",
                "filename": filename,
            },
        },
    ]

def build_base64_dataframe_content(
    prompt: str,
    *,
    dataframe_base64: str,
    filename: str = "movimenti.csv",
    mime_type: str = "text/csv",
) -> list[dict[str, Any]]:
    dataframe_text = decode_base64_text(dataframe_base64)
    return [
        {"type": "text", "text": prompt},
        {
            "type": "text",
            "text": f"Contenuto del DataFrame ({filename}, {mime_type}):\n\n{dataframe_text}",
        },
    ]

TEST_IMAGE_PATH = PROJECT_ROOT / "data" / "foto_1.jpeg"
TEST_PDF_PATH = PROJECT_ROOT / "data" / "operazioni_1.pdf"

test_image_base64 = encode_file_to_base64(TEST_IMAGE_PATH)
test_pdf_base64 = encode_file_to_base64(TEST_PDF_PATH)
sample_dataframe_base64 = base64.b64encode(
    """data,descrizione,importo,categoria,note
2026-04-01,Affitto casa,-950,Affitto o Mutuo,Bonifico mensile
2026-04-02,Spesa supermercato,-84.5,Alimentazione,Esselunga
2026-04-03,Stipendio,2450,Entrata,Accredito""".encode("utf-8")
).decode("utf-8")

In [ ]:
from collections.abc import AsyncIterator, Iterator
from typing import Any, TypeAlias

from langchain_core.messages import AIMessage, AIMessageChunk

MessageContent: TypeAlias = str | list[dict[str, Any]]

def _iter_message_blocks(message: AIMessage | AIMessageChunk) -> Iterator[tuple[str, str, str | None]]:
    saw_structured_block = False

    for block in message.content_blocks:
        if not isinstance(block, dict):
            continue

        saw_structured_block = True
        block_type = block.get("type")
        phase = block.get("phase") if isinstance(block.get("phase"), str) else None

        if block_type == "reasoning":
            reasoning = block.get("reasoning")
            if reasoning:
                yield "reasoning", reasoning, phase
        elif block_type == "text":
            text = block.get("text")
            if text:
                yield "text", text, phase

    if saw_structured_block:
        return

    content = message.content
    if isinstance(content, str):
        if content:
            yield "text", content, None
        return

    if not isinstance(content, list):
        return

    for block in content:
        if isinstance(block, str):
            if block:
                yield "text", block, None
            continue

        if not isinstance(block, dict):
            continue

        block_type = block.get("type")
        phase = block.get("phase") if isinstance(block.get("phase"), str) else None

        if block_type == "reasoning":
            reasoning = block.get("reasoning")
            if reasoning:
                yield "reasoning", reasoning, phase
        elif block_type == "text":
            text = block.get("text")
            if text:
                yield "text", text, phase

def _build_agent_input(user_content: MessageContent) -> dict[str, Any]:
    return {"messages": [{"role": "user", "content": user_content}]}

async def invoke_agent(agent: Any, user_content: MessageContent) -> dict[str, Any]:
    return await agent.ainvoke(_build_agent_input(user_content))

async def stream_agent_events(agent: Any, user_content: MessageContent) -> AsyncIterator[dict[str, str]]:
    async for chunk in agent.astream(
        _build_agent_input(user_content),
        stream_mode="messages",
        subgraphs=True,
        version="v2",
    ):
        if chunk["type"] != "messages":
            continue

        message, _metadata = chunk["data"]
        if not isinstance(message, (AIMessage, AIMessageChunk)):
            continue

        is_subagent = any(namespace.startswith("tools:") for namespace in chunk["ns"] )

        for block_type, text, phase in _iter_message_blocks(message):
            if block_type == "reasoning" or phase == "commentary" or is_subagent:
                event_type = "reasoning"
            else:
                event_type = "answer"

            yield {
                "type": event_type,
                "content": text,
            }

In [ ]:
image_content = build_base64_image_content(
    "Analizza questa immagine, estrai eventuali movimenti bancari e salva quelli validi usando aggiungi_movimenti.",
    image_base64=test_image_base64,
    mime_type=_guess_mime_type(TEST_IMAGE_PATH, "image/jpeg"),
)

async for event in stream_agent_events(agent, image_content):
    print(event)

In [ ]:
pdf_content = build_base64_file_content(
    "Analizza questo PDF, estrai eventuali movimenti bancari e salva quelli validi usando aggiungi_movimenti.",
    file_base64=test_pdf_base64,
    filename=TEST_PDF_PATH.name,
    mime_type=_guess_mime_type(TEST_PDF_PATH, "application/pdf"),
)

dataframe_content = build_base64_dataframe_content(
    "Leggi questo DataFrame serializzato in CSV base64 e salva i movimenti presenti.",
    dataframe_base64=sample_dataframe_base64,
    filename="movimenti.csv",
    mime_type="text/csv",
)

await invoke_agent(agent, dataframe_content)